# Cortex — Fitbit Token Bootstrap
Run this notebook to generate a new `fitbit_tokens.json` with all required scopes.
Once complete, copy the token contents into your `FITBIT_TOKENS` GitHub Secret.

In [ ]:
# ── 1. PASTE YOUR CREDENTIALS HERE ──────────────────────────
FITBIT_CLIENT_ID     = "YOUR_CLIENT_ID"
FITBIT_CLIENT_SECRET = "YOUR_CLIENT_SECRET"
# ─────────────────────────────────────────────────────────────

In [ ]:
import base64, hashlib, secrets, json, time, webbrowser
import requests
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse, parse_qs, urlencode

AUTH_URL     = "https://www.fitbit.com/oauth2/authorize"
TOKEN_URL    = "https://api.fitbit.com/oauth2/token"
REDIRECT_URI = "http://localhost:8080/callback"
SCOPES       = "sleep heartrate activity oxygen_saturation cardio_fitness respiratory_rate profile"

# Generate PKCE challenge
verifier  = secrets.token_urlsafe(64)[:128]
challenge = base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b"=").decode()
state     = secrets.token_urlsafe(16)

params = {
    "client_id":             FITBIT_CLIENT_ID,
    "response_type":         "code",
    "scope":                 SCOPES,
    "redirect_uri":          REDIRECT_URI,
    "code_challenge":        challenge,
    "code_challenge_method": "S256",
    "state":                 state,
}

url = f"{AUTH_URL}?{urlencode(params)}"
print("Opening browser to authorize Fitbit...")
print(f"If it doesn't open automatically, go to:\n{url}")
webbrowser.open(url)

# Catch the callback
auth_code = [None]

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        qs = parse_qs(urlparse(self.path).query)
        if "code" in qs:
            auth_code[0] = qs["code"][0]
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b"Authorized! You can close this tab.")
    def log_message(self, *args):
        pass  # suppress server logs

print("Waiting for Fitbit authorization...")
HTTPServer(("localhost", 8080), Handler).handle_request()
print("Authorization received.")

In [ ]:
# Exchange code for tokens
creds = base64.b64encode(f"{FITBIT_CLIENT_ID}:{FITBIT_CLIENT_SECRET}".encode()).decode()

resp = requests.post(TOKEN_URL, headers={
    "Authorization": f"Basic {creds}",
    "Content-Type":  "application/x-www-form-urlencoded",
}, data={
    "code":          auth_code[0],
    "grant_type":    "authorization_code",
    "redirect_uri":  REDIRECT_URI,
    "code_verifier": verifier,
})

resp.raise_for_status()
tokens = resp.json()
tokens["saved_at"] = time.time()

with open("fitbit_tokens.json", "w") as f:
    json.dump(tokens, f, indent=2)

print("✅ Tokens saved to fitbit_tokens.json")
print(f"   Scopes granted: {tokens.get('scope')}")

In [ ]:
# Print the token contents to copy into GitHub Secrets
print("Copy everything below into your FITBIT_TOKENS GitHub Secret:\n")
print(json.dumps(tokens, indent=2))